In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url = "https://api.groq.com/openai/v1"
)

In [24]:
def llm(instructions, user_prompt, model ="openai/gpt-oss-20b" ):
    message_history = [
        {"role" : "developer", "content": instructions},
        {"role" : "user", "content": user_prompt}
    ]
    
    response = openai_client.responses.create(
        # model="qwen/qwen3.8-27b",
        model=model,
        input=message_history
    )

    return response.output_text


In [13]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

question = "I just discovered the course. Can I join now?"

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)

In [14]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [15]:
from minsearch import Index

index = Index(
    text_fields=["questions", "sections", "answer"],
    keyword_fields=["course"]
)
index.fit(documents)

question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question":2.0, "section":0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
search_results


#wrapping it in a function
def search(question, course = "llm-zoomcamp"):
    boost_dict = {"question":2.0, "section":0.5}
    filter_dict = {"course" : course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [16]:
[doc["question"] for doc in search_results]

['Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Do I have to use OpenAI, or can I use a different provider?',
 'Can I run the course locally instead of Codespaces?',
 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
 'Why do the matrix and for-loop versions of vector search give slightly different results?']

# Building the Prompt

In [19]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question: 
{question}

Context: 
{context}
"""

def build_context(search_results):
    lines =[]

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

prompt = build_prompt(question, search_results)
print(prompt)

Question: 
I just discovered the course. Can I join now?

Context: 
General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Module 1: RAG
Q: Do I have to use OpenAI, or can I use a different provider?
A: If the provider used in the course isn't available or is blocked in your region (or you simply prefer another one), you can use any other LLM provider — the course isn't tied to OpenAI. Just switch to something else:

- Hosted, OpenAI-compatible providers — e.g. Groq, OpenRouter, DeepSeek, Gemini, Z.ai, Mistral. The course code uses the OpenAI client, so you usually only need to change the `base_url`, the API key, and the model name.
- Open models via Hugging

In [27]:
def rag(query, model = "openai/gpt-oss-20b"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model)
    return answer

rag("I Just discovered the course. Can I join now?")
rag("How do I get a Certificate")

'**How to earn a certificate for the course**\n\n1. **Join a live cohort**  \n   - Certificates are only awarded to participants in a *live* cohort (not in self‑paced mode alone).\n\n2. **Complete the capstone project**  \n   - Prepare and submit your capstone while the cohort is still accepting submissions.  \n   - The project is the sole requirement for a certificate; homework is optional.\n\n3. **Finish the required peer reviews**  \n   - You must complete the peer‑review tasks that accompany the capstone.  \n\n4. **Submit before the cohort closes**  \n   - Your project and reviews must be in while the cohort is still accepting them; otherwise you’ll miss the opportunity for a certificate.\n\nOnce these steps are fulfilled, the platform will generate a certificate. The name you enter for the certificate will appear only on that certificate (not publicly). The public leaderboard uses a separate anonymous nickname.'